In [2]:
from google.colab import userdata
import os

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("HUGGINGFACEHUB_API_TOKEN")
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
%cd "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"

/content/drive/.shortcut-targets-by-id/1Yk-plf3-NMUk7VUGN3mnz81xvzAIAyKX/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full


In [5]:
# 1 — Installation des dépendances
!pip install -q \
    transformers \
    sentence-transformers \
    faiss-cpu \
    pymupdf \
    beautifulsoup4 \
    accelerate


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 100.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 111.8 MB/s eta 0:00:00


In [6]:
# =========================
# Reranker — CrossEncoder
# =========================

from sentence_transformers import CrossEncoder
import torch

print("🔁 Initialisation du reranker (CrossEncoder)...")

def load_reranker(model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
    """
    Charge un reranker CrossEncoder avec gestion automatique GPU / CPU.
    Retourne None si le chargement échoue (fallback propre).
    """

    try:
        device = "cuda" if USE_GPU and torch.cuda.is_available() else "cpu"

        reranker = CrossEncoder(
            model_name,
            device=device,
            max_length=512    # 🔒 limite mémoire, suffisant pour QA
        )

        print(f"✅ Reranker chargé sur {device.upper()} : {model_name}")
        return reranker

    except Exception as e:
        print("⚠️ Impossible de charger le reranker.")
        print(f"🛠️ Détail : {e}")
        print("➡️ Le système continuera sans reranking (FAISS seul).")
        return None


# Chargement effectif
reranker = load_reranker()

🔁 Initialisation du reranker (CrossEncoder)...
⚠️ Impossible de charger le reranker.
🛠️ Détail : name 'USE_GPU' is not defined
➡️ Le système continuera sans reranking (FAISS seul).


In [7]:
# =========================
# 2 — Imports & paramètres globaux
# =========================

# ----- Standard library -----
import os
import sys
import pickle
from typing import List, Dict

# ----- Calcul scientifique -----
import numpy as np

# ----- GPU / Deep Learning -----
import torch

# ----- Traitement documents -----
import fitz  # PyMuPDF (PDF)
from bs4 import BeautifulSoup  # HTML

# ----- Recherche vectorielle -----
import faiss

# ----- Embeddings & reranking -----
from sentence_transformers import SentenceTransformer, CrossEncoder

# ----- LLM / Hugging Face -----
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline
)

# =========================
# Paramètres globaux
# =========================

# Mode debug (logs détaillés)
DEBUG = False

# Seuils & constantes de sécurité
MIN_TEXT_LENGTH = 50         # Ignore les textes trop courts
MIN_QUERY_LENGTH = 3         # Ignore les questions trop vagues

# Mémoire & performances
DEFAULT_BATCH_GPU = 8
DEFAULT_BATCH_CPU = 4

# Répertoire de travail
PROJECT_ROOT = "/content"
FAISS_INDEX_DIR = os.path.join(PROJECT_ROOT, "faiss_index_cnrs")

# Création du dossier FAISS si nécessaire
os.makedirs(FAISS_INDEX_DIR, exist_ok=True)

print("✅ Imports et paramètres globaux chargés.")# =========================
# 1 — Installation des dépendances (robuste)
# =========================

# Mise à jour pip (important sur Colab)
!pip install -U pip

# Dépendances principales (versions stables et testées)
!pip install -q \
    torch \
    transformers==4.39.3 \
    sentence-transformers==2.6.1 \
    accelerate==0.27.2 \
    faiss-cpu==1.7.4 \
    pymupdf==1.23.8 \
    beautifulsoup4==4.12.3


✅ Imports et paramètres globaux chargés.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 26.9 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: Could not find a version that satisfies the requirement faiss-cpu==1.7.4 (from versions: 1.8.0, 1.8.0.post1, 1.9.0, 1.9.0.post1, 1.10.0, 1.11.0, 1.11.0.post1, 1.12.0, 1.13.0, 1.13.1, 1.13.2)
ERROR: No matching distribution found for faiss-cpu==1.7.4


In [8]:
# =========================
# PARAMÈTRES (VERSION STABLE)
# =========================

DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"

MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
load_in_4bit = True

EMBEDDING_MODEL = "sentence-transformers/all-mpnet-base-v2"

# Chunking (OK pour JSON structuré)
CHUNK_SIZE = 400
OVERLAP = 100

# Retrieval
TOP_K = 8
SIMILARITY_THRESHOLD = 0.30

# 🔒 Sécurité mémoire LLM
MAX_CHUNKS_FOR_LLM = 3        # ⬅️ CRITIQUE
MAX_CONTEXT_CHARS = 700       # ⬅️ réduit (anti-OOM)

# Génération
MAX_NEW_TOKENS = 200
TEMPERATURE = 0.2
REPETITION_PENALTY = 1.1


In [9]:
# =========================
# 3 — Détection GPU automatique
# =========================

import torch

def can_use_gpu(min_free_gb: float = 4.0, verbose: bool = True) -> bool:
    """
    Détermine si le GPU peut être utilisé en toute sécurité.

    Paramètres :
    - min_free_gb : mémoire GPU libre minimale requise (en Go)
    - verbose : affiche des informations détaillées

    Retour :
    - True si le GPU est utilisable
    - False sinon (fallback CPU)
    """

    # 1️⃣ CUDA disponible ?
    if not torch.cuda.is_available():
        if verbose:
            print("⚠️ CUDA non disponible → mode CPU")
        return False

    try:
        # 2️⃣ Mémoire GPU disponible
        free_bytes, total_bytes = torch.cuda.mem_get_info()
        free_gb = free_bytes / (1024 ** 3)
        total_gb = total_bytes / (1024 ** 3)

        # 3️⃣ Informations GPU
        device_name = torch.cuda.get_device_name(0)

        if verbose:
            print(f"🖥️ GPU détecté : {device_name}")
            print(f"💾 Mémoire GPU libre : {free_gb:.2f} Go / {total_gb:.2f} Go")

        # 4️⃣ Seuil minimal requis
        if free_gb < min_free_gb:
            if verbose:
                print(
                    f"⚠️ Mémoire GPU insuffisante "
                    f"(min requis : {min_free_gb} Go) → mode CPU"
                )
            return False

        return True

    except Exception as e:
        if verbose:
            print(f"❌ Erreur lors de la détection GPU : {e}")
            print("➡️ Fallback CPU")
        return False


# =========================
# Sélection automatique du mode
# =========================

USE_GPU = can_use_gpu(min_free_gb=4.0)

print(
    f"\n🔍 Mode sélectionné : "
    f"{'🟢 GPU' if USE_GPU else '🟡 CPU'}\n"
)


🖥️ GPU détecté : Tesla T4
💾 Mémoire GPU libre : 14.64 Go / 14.74 Go

🔍 Mode sélectionné : 🟢 GPU



In [10]:
# =========================
# 4 — Chargement et préparation des documents
#     (DOSSIER JSON STRUCTURÉ – VERSION FINALE)
# =========================

import os
import json
import re


# ---------- Nettoyage ----------
def clean_text(text: str) -> str:
    text = text.replace("\x00", "")
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{2,}", "\n", text)
    return text.strip()


# ---------- Normalisation ----------
def normalize_text(text: str) -> str:
    text = text.replace("•", "- ")
    text = text.replace("–", "- ")
    text = text.replace("—", "- ")
    text = text.replace("* ", "- ")
    return text


# ---------- Construction de chunks logiques ----------
def build_chunks_from_poste(base_meta: dict, poste: dict):
    """
    Crée des chunks séparés pour mission, activités, compétences, contexte.
    """
    chunks = []

    def add_chunk(label, content):
        if content and len(content.strip()) > 50:
            chunks.append({
                "text": f"{label} : {normalize_text(clean_text(content))}",
                **base_meta
            })

    add_chunk("Mission", poste.get("mission"))
    add_chunk("Activités", poste.get("activites"))
    add_chunk("Compétences", poste.get("competences"))
    add_chunk("Contexte", poste.get("contexte"))

    return chunks


# =========================
# CONSTRUCTION DU CORPUS
# =========================

documents = []

print("📂 Chargement des documents JSON structurés...")

for filename in os.listdir(DATA_DIR):
    if not filename.lower().endswith(".json"):
        continue

    path = os.path.join(DATA_DIR, filename)

    try:
        with open(path, "r", encoding="utf-8") as f:
            data = json.load(f)

        # Métadonnées communes au concours
        base_meta_common = {
            "source": filename,
            "bap": data.get("bap"),
            "grade": data.get("grade"),
            "concours_label": data.get("concours_label"),
            "concours_num": data.get("concours_num"),
            "nb_postes": data.get("nb_postes"),
        }

        # Boucle sur les postes
        for poste in data.get("postes", []):
            base_meta = {
                **base_meta_common,
                "poste_num": poste.get("poste_num"),
                "affectation": poste.get("affectation"),
                "groupe_fonction": poste.get("groupe_fonction"),
            }

            chunks = build_chunks_from_poste(base_meta, poste)
            documents.extend(chunks)

    except Exception as e:
        print(f"⚠️ Erreur JSON {filename} : {e}")

print(f"✅ Documents indexés : {len(documents)} chunks")


📂 Chargement des documents JSON structurés...
✅ Documents indexés : 592 chunks


In [11]:
# =========================
# 5 — Embeddings & FAISS
# =========================

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
import os
import torch

print("🔎 Initialisation du modèle d'embeddings...")

# 1️⃣ Chargement du modèle d'embeddings (léger et stable)
embedder = SentenceTransformer(
    "sentence-transformers/all-mpnet-base-v2",
    device="cuda" if USE_GPU and torch.cuda.is_available() else "cpu"
)

# 2️⃣ Préparation des textes (sécurité)
texts = [d["text"] for d in documents if d.get("text")]

if not texts:
    raise ValueError("❌ Aucun texte valide pour la vectorisation.")

# 3️⃣ Encodage par batch (évite les pics mémoire)
print(f"📐 Calcul des embeddings pour {len(texts)} chunks...")

embeddings = embedder.encode(
    texts,
    batch_size=8 if USE_GPU else 4,      # 🔑 batch réduit pour stabilité
    normalize_embeddings=True,
    show_progress_bar=True
)

# 4️⃣ Conversion FAISS (float32 obligatoire)
embeddings = np.asarray(embeddings, dtype="float32")

# 5️⃣ Création de l'index FAISS (cosine similarity via IP)
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)

# Sécurité : vérifier cohérence
assert index.is_trained, "Index FAISS non entraîné"

# 6️⃣ Ajout des vecteurs
index.add(embeddings)

print(f"✅ Index FAISS créé avec {index.ntotal} vecteurs.")



🔎 Initialisation du modèle d'embeddings...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

📐 Calcul des embeddings pour 592 chunks...


Batches:   0%|          | 0/74 [00:00<?, ?it/s]

✅ Index FAISS créé avec 592 vecteurs.


In [12]:
def handle_small_talk(question: str):
    q = question.lower().strip()

    greetings = ["bonjour", "bonsoir", "salut", "hello", "hi"]
    thanks = ["merci", "merci beaucoup", "thanks"]

    if q in greetings:
        return (
            "Bonjour 👋\n"
            "Je suis l’agent d’information sur les concours ingénieurs du CNRS.\n"
            "Vous pouvez me poser des questions sur les concours, postes, missions ou compétences."
        )

    if q in thanks:
        return "Avec plaisir 😊 N’hésitez pas si vous avez d’autres questions sur les concours CNRS."

    return None


In [13]:
from collections import Counter

def filter_same_source(chunks):
    """
    Garde uniquement les chunks provenant majoritairement
    du même fichier (même concours).
    """
    if not chunks:
        return []

    sources = [c["source"] for c in chunks]
    main_source = Counter(sources).most_common(1)[0][0]

    return [c for c in chunks if c["source"] == main_source]


In [14]:
def is_orientation_question(question: str) -> bool:
    keywords = [
        "correspond", "profil", "je suis", "quel concours",
        "orienter", "adapté", "accessible avec"
    ]
    q = question.lower()
    return any(k in q for k in keywords)


In [15]:
# =========================
# 6 — Retrieval avec reranking
# =========================

def retrieve(question: str):
    """
    Recherche les passages les plus pertinents pour une question donnée.
    Étapes :
    1. Retrieval large via FAISS (rappel)
    2. Filtrage par seuil de similarité
    3. Reranking précis (cross-encoder)
    4. Sélection finale TOP_K
    """

    # 🔒 Sécurité minimale
    if not question or len(question.strip()) < 3:
        return []

    # 1️⃣ Embedding de la question
    try:
        q_emb = embedder.encode(
            [question],
            normalize_embeddings=True
        )
    except Exception:
        return []

    # 2️⃣ Retrieval large (on sur-échantillonne pour le reranker)
    try:
        scores, indices = index.search(q_emb, TOP_K * 3)
    except Exception:
        return []

    # 3️⃣ Filtrage par similarité FAISS
    candidates = []
    for score, idx in zip(scores[0], indices[0]):

        # Index invalide
        if idx < 0 or idx >= len(documents):
            continue

        # Seuil anti-hallucination
        if score < SIMILARITY_THRESHOLD:
            continue

        doc = documents[idx].copy()
        doc["faiss_score"] = float(score)
        candidates.append(doc)

    # Aucun contexte fiable
    if not candidates:
        return []

    # 4️⃣ Reranking précis (si disponible)
    if reranker is not None:
        try:
            pairs = [(question, c["text"]) for c in candidates]
            rerank_scores = reranker.predict(pairs)

            for c, r_score in zip(candidates, rerank_scores):
                c["rerank_score"] = float(r_score)

            # Combinaison FAISS + reranker
            reranked = sorted(
                candidates,
                key=lambda c: (c["rerank_score"], c["faiss_score"]),
                reverse=True
            )
        except Exception:
            # Fallback : FAISS seul
            reranked = sorted(
                candidates,
                key=lambda c: c["faiss_score"],
                reverse=True
            )
    else:
        reranked = sorted(
            candidates,
            key=lambda c: c["faiss_score"],
            reverse=True
        )

    # 5️⃣ Sélection finale (TOP_K)
    return reranked[:TOP_K]


In [16]:
# =========================
# 7 — Chargement du modèle LLM (Qwen/Qwen2.5-14B-Instruct)
# =========================

from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
import os

print("🚀 Chargement du modèle LLM...")

# 1️⃣ Chargement du tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    use_fast=True
)

# 2️⃣ Chargement du modèle avec gestion GPU / CPU
if USE_GPU and torch.cuda.is_available():

    print("🟢 GPU détecté — Chargement en mode GPU optimisé")

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",                  # Placement automatique GPU / CPU
        torch_dtype=torch.float16,          # Réduction mémoire
        low_cpu_mem_usage=True,
        max_memory={
            0: "14GiB",                     # GPU T4 ≈ 16 Go
            "cpu": "10GiB"
        }
    )

else:
    print("🟡 GPU non disponible — Chargement en mode CPU (secours)")

    os.environ["CUDA_VISIBLE_DEVICES"] = ""
    torch.set_num_threads(4)

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map={"": "cpu"},
        torch_dtype=torch.float32,
        low_cpu_mem_usage=True
    )

# 3️⃣ Mise en mode évaluation (important)
model.eval()

# 4️⃣ Pipeline de génération STRICT (anti-hallucination)
pipe = pipeline(
    task="text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=MAX_NEW_TOKENS,

    # 🔑 CLÉ DU SUCCÈS
    do_sample=False,        # Toujours déterministe
    temperature=0.2,        # ⚠️ PAS 0.0
    top_p=1.0,              # On ne tronque plus le raisonnement

    repetition_penalty=1.1, # Évite le blabla

    return_full_text=False,
    pad_token_id=tokenizer.eos_token_id
)

print("✅ Modèle prêt pour l'inférence.")


🚀 Chargement du modèle LLM...


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/3.86G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


✅ Modèle prêt pour l'inférence.


In [17]:
# =========================
# 8 — Prompt CNRS STRICT (anti-hallucination)
# =========================

def build_prompt(question: str, contexts: list) -> str:
    context_text = "\n".join(
        f"- {c['text']}"
        for c in contexts
    )

    return f"""
Tu es un agent officiel d'information sur les concours ingénieurs du CNRS.

RÈGLES STRICTES :
- Utilise UNIQUEMENT les informations présentes dans le CONTEXTE.
- Ne déduis rien.
- Ne complète rien.
- Ne donne aucune information absente du CONTEXTE.
- Si l'information demandée n'est pas clairement présente, répond EXACTEMENT :
  "{REFUS}"
- Répond uniquement en français.
- Ne pose pas de questions.

CONTEXTE :
{context_text}

QUESTION :
{question}

RÉPONSE :
"""
# #################################################################################

def build_orientation_prompt(question, contexts):
    context_text = "\n".join(
        f"- {c['text']}"
        for c in contexts
    )

    return f"""
Tu es un agent d'information du CNRS.

Tu dois aider à ORIENTER un candidat en comparant son profil
avec les missions et compétences décrites dans les documents.

RÈGLES :
- Tu ne dois PAS affirmer une correspondance certaine.
- Tu dois utiliser des formulations prudentes :
  "peut correspondre", "est cohérent avec", "semble en adéquation".
- Tu dois expliquer SUR QUELS ÉLÉMENTS tu te bases.
- Tu dois citer les sources utilisées.
- Si aucune correspondance raisonnable n'est possible, dis-le clairement.

DOCUMENTS :
{context_text}

QUESTION :
{question}

RÉPONSE :
"""

################################################################################
def build_search_query(question: str) -> str:
    """
    Transforme une question d'orientation en requête documentaire exploitable.
    """
    q = question.lower()

    keywords = []
    if "statistique" in q or "data" in q:
        keywords.append("analyse de données")
    if "biologique" in q or "bio" in q:
        keywords.append("biologie")
    if "informatique" in q:
        keywords.append("informatique")
    if "master" in q or "bac+5" in q:
        keywords.append("ingénieur")
    if "omics" in q:
        keywords.append("omiques")

    # fallback minimal
    if not keywords:
        return question

    return " ".join(keywords)


In [18]:
# =========================
# 9 — Fonction answer()
# =========================
REFUS = "Je ne dispose pas de cette information dans les documents de référence."

def answer(question: str):
    # 1️⃣ Small talk
    st = handle_small_talk(question)
    if st:
        return {"response": st, "sources": [], "has_answer": True}

    # 2️⃣ Détection orientation
    orientation = is_orientation_question(question)

    # 3️⃣ Construction requête FAISS
    search_query = build_search_query(question) if orientation else question

    # 4️⃣ Retrieval
    contexts = retrieve(search_query)

    if not contexts:
        return {"response": REFUS, "sources": [], "has_answer": False}

    # 5️⃣ Filtrage par concours
    contexts = filter_same_source(contexts)

    # 6️⃣ Choix du prompt
    if orientation:
        prompt = build_orientation_prompt(question, contexts)
    else:
        prompt = build_prompt(question, contexts)

    # 7️⃣ Génération
    output = pipe(prompt)[0]["generated_text"].strip()

    # 8️⃣ Refus
    if REFUS.lower() in output.lower():
        return {"response": REFUS, "sources": [], "has_answer": False}

    # 9️⃣ Sources propres
    sources = sorted({f"{c['source']} | page {c['page']}" for c in contexts})

    return {"response": output, "sources": sources, "has_answer": True}


In [19]:
def print_answer(result):
    print("\n📘 RÉPONSE :\n")
    print(result["response"])

    print("\n📂 SOURCES :")
    if not result["sources"]:
        print("Aucune")
    else:
        for s in result["sources"]:
            print(f"- {s}")


In [20]:
def select_contexts_for_llm(contexts, max_chunks=3):
    """
    Sélectionne intelligemment les chunks à envoyer au LLM
    pour éviter les OOM et maximiser la pertinence.
    Priorité : Mission > Compétences > Contexte > Autres
    """
    if not contexts:
        return []

    priority_order = ["Mission", "Compétences", "Contexte", "Activités"]

    selected = []

    for label in priority_order:
        for c in contexts:
            if label.lower() in c["text"].lower() and c not in selected:
                selected.append(c)
                break
        if len(selected) >= max_chunks:
            break

    # Fallback : compléter si pas assez de chunks
    if len(selected) < max_chunks:
        for c in contexts:
            if c not in selected:
                selected.append(c)
            if len(selected) >= max_chunks:
                break

    return selected[:max_chunks]


In [21]:
def answer(question: str):
    # 1️⃣ Small talk
    st = handle_small_talk(question)
    if st:
        return {
            "response": st,
            "sources": [],
            "has_answer": True
        }

    # 2️⃣ Détection orientation
    orientation = is_orientation_question(question)

    # 3️⃣ Requête de recherche adaptée
    search_query = build_search_query(question) if orientation else question

    # 4️⃣ Retrieval
    contexts = retrieve(search_query)

    if not contexts:
        return {
            "response": REFUS,
            "sources": [],
            "has_answer": False
        }

    # 5️⃣ 🔒 LIMITATION INTELLIGENTE DU CONTEXTE (ANTI-OOM)
    contexts = select_contexts_for_llm(
        contexts,
        max_chunks=MAX_CHUNKS_FOR_LLM
    )

    # 6️⃣ Construction du prompt
    if orientation:
        prompt = build_orientation_prompt(question, contexts)
    else:
        prompt = build_prompt(question, contexts)

    # 7️⃣ Génération
    try:
        output = pipe(prompt)[0]["generated_text"].strip()
    except RuntimeError as e:
        if "out of memory" in str(e).lower():
            torch.cuda.empty_cache()
            return {
                "response": "Erreur mémoire GPU. Merci de reformuler la question.",
                "sources": [],
                "has_answer": False
            }
        raise e

    # 8️⃣ Détection refus
    if REFUS.lower() in output.lower():
        return {
            "response": REFUS,
            "sources": [],
            "has_answer": False
        }

    # 9️⃣ Sources propres et dédupliquées
    sources = sorted({
        f"{c['source']} | poste {c.get('poste_num')} | {c.get('affectation')}"
        for c in contexts
    })

    return {
        "response": output,
        "sources": list(sources),
        "has_answer": True
    }


In [23]:
print("\n" + "=" * 70)
print("🤖 Agent conversationnel CNRS – Concours Ingénieurs")
print("📌 Basé uniquement sur les documents officiels fournis")
print("✋ Tapez 'stop', 'quit' ou 'exit' pour quitter")
print("=" * 70 + "\n")

while True:
    try:
        q = input("❓ Votre question : ").strip()

        if not q:
            print("⚠️ Merci de poser une question.\n")
            continue

        if q.lower() in ["stop", "quit", "exit"]:
            print("\n👋 Fin de la session. Merci !")
            break

        print("\n🔎 Analyse de la question...\n")

        # 🔹 APPEL CORRECT DE answer()
        result = answer(q)

        # 🔹 AFFICHAGE PROPRE
        print_answer(result)

        print("\n" + "-" * 70)

    except KeyboardInterrupt:
        print("\n\n👋 Interruption utilisateur. Fin de la session.")
        break

    except Exception as e:
        print(f"\n❌ Erreur inattendue : {e}")
        print("\n" + "-" * 70)


🤖 Agent conversationnel CNRS – Concours Ingénieurs
📌 Basé uniquement sur les documents officiels fournis
✋ Tapez 'stop', 'quit' ou 'exit' pour quitter


🔎 Analyse de la question...


📘 RÉPONSE :

Erreur mémoire GPU. Merci de reformuler la question.

📂 SOURCES :
Aucune

----------------------------------------------------------------------
❓ Votre question : stop

👋 Fin de la session. Merci !


In [22]:
#génération de réponses
from google.colab import files
import pandas as pd
import io

# 1. Fenêtre d'upload
uploaded = files.upload()

# 2. Récupération du nom du fichier
file_name = list(uploaded.keys())[0]

# 3. Chargement dans un DataFrame (on gère le séparateur virgule ou point-virgule)
df = pd.read_csv(io.BytesIO(uploaded[file_name]), sep=None, engine='python')

print("Fichier chargé avec succès !")

# 4. Lancement de votre fonction answer
from tqdm import tqdm
tqdm.pandas()
df['model_answer'] = df['Question'].progress_apply(lambda x: answer(str(x)))

# 5. Téléchargement automatique du résultat
output_file = "resultats_chatbot.csv"
df.to_csv(output_file, index=False, sep=';', encoding='utf-8-sig')
files.download(output_file)

Saving jeu_test.csv to jeu_test.csv
Fichier chargé avec succès !


100%|██████████| 48/48 [10:52<00:00, 13.60s/it]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>